In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    recall_score,
    precision_score,
    classification_report,
    confusion_matrix,
    f1_score
)

In [3]:
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

from catboost import CatBoostClassifier

train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

In [4]:
    df["X13_X35_ratio"] = df["X13"] / (df["X35"] + 1)

    df["X35_X36_ratio"] = df["X35"] / (df["X36"] + 1)

    df["X13_minus_X34"] = df["X13"] - df["X34"]

    df["X31_plus_X32"] = df["X31"] + df["X32"]

    df["defect_score"] = (
        (df["X13"] > 1130).astype(int)
        +
        (df["X35"] < 700000).astype(int)
        +
        (df["X36"] < 150).astype(int)
        +
        (df["X34"] < 230).astype(int)
    )

In [5]:
X = train.drop(columns=["CoilID", "Y"])
y = train["Y"].astype(int)

X_test = test.drop(columns=["CoilID"])

In [7]:
    X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
    )

In [8]:
imputer = IterativeImputer(
    random_state=42,
    max_iter=10
)

X_train = imputer.fit_transform(X_train)

X_val = imputer.transform(X_val)

X_test = imputer.transform(X_test)

In [9]:
negative_count = (y_train == 0).sum()

positive_count = (y_train == 1).sum()

class_weight = negative_count / positive_count

print("Class Weight =", class_weight)

Class Weight = 19.39622641509434


In [10]:
model = CatBoostClassifier(

    iterations=1000,

    learning_rate=0.03,

    depth=6,

    loss_function="Logloss",

    eval_metric="Recall",

    class_weights=[1, class_weight],

    random_seed=42,

    verbose=100
)

model.fit(
    X_train,
    y_train
)

0:	learn: 0.8867925	total: 35.5ms	remaining: 35.5s
100:	learn: 1.0000000	total: 945ms	remaining: 8.41s
200:	learn: 1.0000000	total: 1.84s	remaining: 7.32s
300:	learn: 1.0000000	total: 2.75s	remaining: 6.39s
400:	learn: 1.0000000	total: 3.66s	remaining: 5.46s
500:	learn: 1.0000000	total: 4.56s	remaining: 4.54s
600:	learn: 1.0000000	total: 5.47s	remaining: 3.63s
700:	learn: 1.0000000	total: 6.4s	remaining: 2.73s
800:	learn: 1.0000000	total: 7.3s	remaining: 1.81s
900:	learn: 1.0000000	total: 8.21s	remaining: 902ms
999:	learn: 1.0000000	total: 9.1s	remaining: 0us


CatBoostClassifier(class_weights=[1, np.float64(19.39622641509434)], depth=6, eval_metric='Recall', iterations=1000, learning_rate=0.03, loss_function='Logloss', random_seed=42, verbose=100)

In [11]:
val_probs = model.predict_proba(X_val)[:,1]

In [12]:
best_threshold = 0.5

best_recall = 0

best_precision = 0

for threshold in np.arange(0.001, 1.0, 0.001):

    preds = (val_probs >= threshold).astype(int)

    recall = recall_score(
        y_val,
        preds,
        zero_division=0
    )

    precision = precision_score(
        y_val,
        preds,
        zero_division=0
    )

    if recall > best_recall:

        best_recall = recall
        best_precision = precision
        best_threshold = threshold

    elif recall == best_recall and precision > best_precision:

        best_precision = precision
        best_threshold = threshold

In [13]:
final_preds = (
    val_probs >= best_threshold
).astype(int)

print("\n")
print("FINAL VALIDATION RESULTS")

print(
    f"Threshold : {best_threshold:.3f}"
)

print(
    f"Recall    : {best_recall*100:.2f}%"
)

print(
    f"Precision : {best_precision*100:.2f}%"
)

print(
    f"F1 Score  : {f1_score(y_val, final_preds):.4f}"
)

print("\nClassification Report")

print(
    classification_report(
        y_val,
        final_preds
    )
)

print("\nConfusion Matrix")

print(
    confusion_matrix(
        y_val,
        final_preds
    )
)



FINAL VALIDATION RESULTS
Threshold : 0.002
Recall    : 92.31%
Precision : 11.32%
F1 Score  : 0.2017

Classification Report
              precision    recall  f1-score   support

           0       0.99      0.64      0.78       258
           1       0.11      0.92      0.20        13

    accuracy                           0.65       271
   macro avg       0.55      0.78      0.49       271
weighted avg       0.95      0.65      0.75       271


Confusion Matrix
[[164  94]
 [  1  12]]


In [15]:

test_probs = model.predict_proba(X_test)[:,1]

test_preds = (
    test_probs >= best_threshold
).astype(int)

submission = pd.DataFrame({

    "CoilID": test["CoilID"],

    "Y": test_preds
})

submission.to_csv(
    "submission.csv",
    index=False
)

print("\nsubmission.csv created successfully")


submission.csv created successfully
